In [ ]:
import os
import logging
import time
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import csv
import pandas as pd 
import numpy as np 

from eutils import EutilsNCBIError, EutilsRequestError
from metapub import PubMedFetcher, pubmedcentral

In [2]:
API_KEY = "70faf5cc42501a814dcc4bdb1862acaf3909"

In [5]:
# Initialize logger
prefix = "test"+str(datetime.now()).split()[0]
file_handler = logging.FileHandler(f"Results/{prefix}_Examples.log", mode='w')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logging.getLogger().addHandler(file_handler)

In [ ]:
# Initialize PubMed fetcher
fetcher = PubMedFetcher()

In [ ]:

def save_pmids(pmid_array, directory="PMID_lists"):
    """Save PMIDs to both a text file and a NumPy binary file."""
    # Ensure the directory exists
    os.makedirs(directory, exist_ok=True)

    # Create a date tag for the filename
    date_tag = datetime.now().isoformat()[:10]

    # File paths
    txt_file_path = os.path.join(directory, f'pmids_{date_tag}.txt')
    npy_file_path = os.path.join(directory, f'pmids_{date_tag}.npy')

    # Save PMIDs to a text file
    np.savetxt(txt_file_path, pmid_array, fmt='%s', delimiter=",")
    logging.info(f"PMIDs saved to text file: {txt_file_path}")

    # Save PMIDs to a NumPy binary file
    np.save(npy_file_path, pmid_array)
    logging.info(f"PMIDs saved to binary file: {npy_file_path}")


@retry_on_communication_error
def fetch_pmcid(pmid):
    try:
        pmc = pubmedcentral.get_pmcid_for_otherid(pmid)
        return pmc
    except (CommunicationError, ConnectionError) as e:
        logging.error(f"Error: API request failed for {pmid}: {e}")
        return None
    except Exception as e:
        logging.error(f"Unexpected error for {pmid}: {e}")
        return None

def get_pmcid_for_otherid(pmid_clean_list):
    PMCIDs = []
    with ThreadPoolExecutor(max_workers=10) as executor:  # Adjust max_workers based on needs
        future_to_pmid = {executor.submit(fetch_pmcid, pmid): pmid for pmid in pmid_clean_list}
        for future in as_completed(future_to_pmid):
            pmid = future_to_pmid[future]
            try:
                pmc = future.result()
                PMCIDs.append(pmc)
            except Exception as e:
                logging.error(f"Error processing PMID {pmid}: {e}")
                PMCIDs.append(None)
    return PMCIDs

def filter_oa_database(oa_file_list, pmc_ids_filename):
    """
    Filters based on the csv database list of PMCs that are available for full_text mining and writes them  to CSV and txt.

    Parameters:
    oa_file_list (str): Filename of the CSV containing the OA file list.
    pmc_ids_filename (str): Filename of the CSV containing the PMC IDs.
    """


    # Read CSV files
    oa_file_list_df = pd.read_csv(oa_file_list)
    pmc_ids_df = pd.read_csv(pmc_ids_filename)

    # Extract PMC ID list from the DataFrame
    pmc_id_list = pmc_ids_df.iloc[:, 0].tolist()

    # Filter oa_database based on PMC ID list
    filtered_oa_database = oa_file_list_df[oa_file_list_df["Accession ID"].isin(pmc_id_list)]

    # Open a text file to write
    with open('full_text_pmc.txt', 'w') as file:
        for item in filtered_oa_database["Accession ID"]:
            file.write(str(item) + '\n')


    # Save the filtered DataFrame to a new CSV file
    filtered_oa_database.to_csv("full_text_data.csv", index=False)

    return filtered_oa_database


In [ ]:
def main():
    # Step 1: Read the query from a file
    query_file = "query" 
    query = read_query_from_file(query_file)
    if not query:
        logging.error("Query reading failed. Exiting.")
        return

    # Step 2: Fetch PMIDs over a specified period
    start_date = "2000-01-01"
    stop_date = None  # Will default to the current date if None
    pmid_array = fetch_pmids_over_period(query_file, start=start_date, stop=stop_date)
    if pmid_array.size == 0:
        logging.error("No PMIDs fetched. Exiting.")
        return

    # Step 3: Save the PMIDs to files
    save_pmids(pmid_array)

    # Step 4: Retrieve PMCIDs for the fetched PMIDs
    pmc_id_list = get_pmcid_for_otherid(pmid_array)
    pmc_ids_filename = "PMCIDS.csv"
    pd.DataFrame(pmc_id_list, columns=["PMCID"]).to_csv(pmc_ids_filename, index=False)
    logging.info(f"PMCIDs saved to {pmc_ids_filename}")

    # Step 5: Filter the OA database using the retrieved PMCIDs
    oa_file_list = "oa_file_list.csv"  # OA file list CSV filename
    pmc_ids_filename = "PMCIDS.csv"
    filtered_df = filter_oa_database(oa_file_list, pmc_ids_filename)

    logging.info("OA database filtering completed.")
    return filtered_df

# Call the main function to execute the workflow
if __name__ == "__main__":
    main()


In [ ]:
!curl -s "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC10759277/unicode" > "PMC10759277.json"


In [ ]:
%%bash

mkdir -p ./Full_text_jsons

while IFS= read -r PMCID; do
    url="https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/${PMCID}/unicode"
    curl -s "${url}" > "./Full_text_jsons/${PMCID}.json"
done < full_text_pmc.txt



In [ ]:
!powershell -Command "mkdir -p ./Full_text_jsons; Get-Content full_text_pmc.txt | ForEach-Object { Invoke-RestMethod -Uri ('https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/' + $_ + '/unicode') -OutFile ('./Full_text_jsons/' + $_ + '.json') }"

In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from typing import List, Dict
from metapub import PubMedFetcher, PubMedArticle, pubmedcentral

fetcher = PubMedFetcher()

@retry_on_communication_error()
def fetch_article(pmid: str) -> Dict[str, str]:
    """Fetch a single article and return its data as a dict."""
    article = fetcher.article_by_pmid(pmid)
    return {
        "pmid": article.pmid,
        "pmc": article.pmc,
        "title": article.title,
        "journal": article.journal,
        "doi": article.doi,
        "issn": article.issn,
    }

def fetch_articles_to_dataframe(pmids: List[str], workers: int = 5) -> pd.DataFrame:
    """Fetch articles in parallel and return them as a DataFrame."""
    with ThreadPoolExecutor(max_workers=workers) as executor:
        articles_data = list(executor.map(fetch_article, pmids))
    
    return pd.DataFrame(articles_data)

In [ ]:
pmids = ["12345678", "23456789"]  # Example PMIDs
try1 = fetch_articles_to_dataframe(pmids)